Results Figures:

In [ ]:
import json
import ast
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from pathlib import Path
from sklearn.metrics import roc_curve, auc
from scipy.interpolate import pchip_interpolate

# ==========================================
# 1. GLOBAL PUBLICATION STYLE (LaTeX/Serif)
# ==========================================
# Consistent colors: Green for Diverticulitis (#2E7D32), Red for Cancer (#C62828)
COLOR_DIVERT = "#2E7D32" 
COLOR_CANCER = "#C62828"
COLOR_GLOBAL = "#1C4E80" # Deep Blue for Global/ROC

mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 14,
    "axes.labelsize": 16,
    "axes.titlesize": 16,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.linewidth": 1.2,
    "legend.fontsize": 13,
    "mathtext.fontset": "stix",
})

# ==========================================
# 2. METRICS PLOTTING (nnU-Net Results)
# ==========================================
def plot_segmentation_metrics(json_path, class_csv):
    with open(json_path, "r") as f:
        results = json.load(f)
    
    per_case = results["per_case"]
    rows = [{"uid": str(uid), **v["before"]} for uid, v in per_case.items()]
    df = pd.DataFrame(rows)
    
    class_df = pd.read_csv(class_csv)
    class_df["UID"] = class_df["UID"].astype(str)
    df = df.merge(class_df, left_on="uid", right_on="UID", how="left")

    metrics = [("dice", "Dice Coefficient"), ("precision", "Precision"), ("recall", "Recall")]
    
    for metric_key, ylabel in metrics:
        class_data = [df[df["target"] == 0][metric_key], df[df["target"] == 1][metric_key]]
        
        fig, ax = plt.subplots(figsize=(4.5, 6), dpi=100)
        
        bp = ax.boxplot(
            class_data,
            labels=["Diverticulitis", "Colon Cancer"],
            patch_artist=True,
            widths=0.6,
            medianprops=dict(color="black", linewidth=2),
            boxprops=dict(linewidth=1.5)
        )

        # Apply standardized colors
        bp["boxes"][0].set_facecolor(COLOR_DIVERT)
        bp["boxes"][1].set_facecolor(COLOR_CANCER)
        bp["boxes"][0].set_alpha(0.7)
        bp["boxes"][1].set_alpha(0.7)

        # Style tick labels
        for tick, color in zip(ax.get_xticklabels(), [COLOR_DIVERT, COLOR_CANCER]):
            tick.set_color(color)
            tick.set_fontweight("bold")

        ax.set_ylabel(ylabel, fontweight="bold")
        #ax.set_title(f"Segmentation Performance: {ylabel}", pad=15)
        global_mean = df[metric_key].mean()
        ax.text(
            0.15, 1.05, 
            f"Global Mean: {global_mean:.3f}", 
            transform=ax.transAxes, 
            verticalalignment='top',
            fontweight='bold',
            fontsize=14,
            color=COLOR_GLOBAL,
            bbox=dict(
                facecolor='white', 
                edgecolor=COLOR_GLOBAL, 
                boxstyle='round,pad=0.5',
                alpha=0.9,
                linewidth=1.5
            )
        )
        ax.grid(axis='y', linestyle='--', alpha=0.3)
        
        plt.tight_layout()
        plt.show()

# ==========================================
# 3. SMOOTH ROC CURVE (Classification Results)
# ==========================================
def plot_smooth_roc(csv_path):
    df = pd.read_csv(csv_path)
    y_true = df["GT"].values
    y_scores = df["NN_pred"].apply(lambda x: ast.literal_eval(x)[1]).values

    fpr, tpr, _ = roc_curve(y_true, y_scores)
    roc_auc = auc(fpr, tpr)

    # Monotonic smoothing
    _, idx = np.unique(fpr, return_index=True)
    fpr_smooth = np.linspace(0, 1, 200)
    tpr_smooth = np.clip(pchip_interpolate(fpr[idx], tpr[idx], fpr_smooth), 0, 1)

    fig, ax = plt.subplots(figsize=(6, 6), dpi=120)
    ax.fill_between(fpr_smooth, tpr_smooth, alpha=0.1, color=COLOR_GLOBAL)
    ax.plot(
        fpr_smooth, 
        tpr_smooth, 
        color=COLOR_GLOBAL, 
        lw=2.5, 
        label=rf"$\mathbf{{Model\ (AUC = {roc_auc:.3f})}}$",
        zorder=3
    )

    # Reference "Random Chance" Line (Now added to legend)
    ax.plot(
        [0, 1], [0, 1], 
        linestyle="--", 
        color="#888888", 
        lw=1.5, 
        alpha=0.8, 
        label="Random Classifier (AUC = 0.50)",
        zorder=2
    )

    ax.set_xlim(-0.01, 1.0)
    ax.set_ylim(0.0, 1.02)
    ax.set_xlabel("False Positive Rate (1 − Specificity)", labelpad=10, fontdict={"fontweight": "bold"})
    ax.set_ylabel("True Positive Rate (Sensitivity)", labelpad=10, fontdict={"fontweight": "bold"})
    ax.legend(loc="lower right", frameon=False)
    #ax.set_title("Classification ROC Curve", pad=15)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 4. CONFUSION MATRIX (Aligned Labels)
# ==========================================
def plot_consistent_cm():
    cm = np.array([[26, 6], [1, 44]])
    class_names = ["Diverticulitis", "Colon Cancer"]
    
    fig, ax = plt.subplots(figsize=(6, 6), dpi=120)
    im = ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues, alpha=0.8)

    #ax.set_title("Confusion Matrix: Differential Diagnosis", pad=25, fontweight='bold')
    ax.set_ylabel("True Pathology", labelpad=20, fontweight='bold')
    ax.set_xlabel("Predicted Pathology", labelpad=20, fontweight='bold')

    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names, rotation=90, va='center')

    # Colorize labels
    for i, tick in enumerate(ax.get_xticklabels()):
        tick.set_color([COLOR_DIVERT, COLOR_CANCER][i])
        tick.set_fontweight('bold')
    for i, tick in enumerate(ax.get_yticklabels()):
        tick.set_color([COLOR_DIVERT, COLOR_CANCER][i])
        tick.set_fontweight('bold')

    # Cell values
    thresh = cm.max() / 1.5
    for i in range(2):
        for j in range(2):
            ax.text(j, i, format(cm[i, j], 'd'), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black",
                    fontsize=18, fontweight='bold')

    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.1).outline.set_visible(False)
    plt.tight_layout()
    plt.show()
csv_path = (
    "/data/colon_cancer/CC_Detection/Resnet_results/inf_outputs/"
    "ResNet_Dataset110_CC_2026_02_11_172950/Dataset110_CC/results.csv"
)
# ==========================================
# EXECUTION
# ==========================================
#plot_segmentation_metrics(json_path, class_csv) # Uncomment to run
#plot_smooth_roc(csv_path)                      # Uncomment to run
plot_consistent_cm()

Qualitative Results:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nibabel as nib
import matplotlib as mpl
import pandas as pd
from pathlib import Path

# ============================================================
# Global Style Consistency
# ============================================================
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 14,
    "mathtext.fontset": "stix",
})

# Neutral Mask Colors (to distinguish GT from Pred)
COLOR_GT = [1.0, 0.57, 0.0]   # Gold
COLOR_PRED = [0.0, 0.69, 1.0]  # Blue

# Pathology Label Colors (Consistent with Stats/CM)
PATH_COLORS = {0: "#2E7D32", 1: "#C62828"} # 0: Div (Green), 1: Cancer (Red)
PATH_NAMES = {0: "Diverticulitis", 1: "Colon Cancer"}

# ============================================================
# Helper Functions (Windowing & BBox)
# ============================================================

def load_volume(path):
    path = Path(path)
    if path.suffix == ".npy": return np.load(path)
    return nib.load(str(path)).get_fdata()

def load_case(uid, img_dir, gt_dir, pred_dir):
    img = load_volume(Path(img_dir) / f"{uid}_0000.nii.gz")
    gt = load_volume(Path(gt_dir) / f"{uid}.nii.gz")
    pred = load_volume(Path(pred_dir) / f"{uid}.nii.gz")
    return img, gt, pred

def window_ct_hu(ct, level=50, width=350):
    lo, hi = level - width / 2, level + width / 2
    return (np.clip(ct, lo, hi) - lo) / (hi - lo + 1e-6)

def compute_square_bbox(mask, axis, margin=60):
    coords = np.where(mask > 0)
    axes_2d = [i for i in range(3) if i != axis]
    y_min, y_max = coords[axes_2d[0]].min(), coords[axes_2d[0]].max()
    x_min, x_max = coords[axes_2d[1]].min(), coords[axes_2d[1]].max()
    size = max(y_max - y_min, x_max - x_min) + margin
    cy, cx = (y_min + y_max) // 2, (x_min + x_max) // 2
    return max(0, cy - size // 2), cy + size // 2, max(0, cx - size // 2), cx + size // 2

def crop_and_rotate(slice2d, bbox):
    y0, y1, x0, x1 = bbox
    return np.rot90(slice2d[y0:y1, x0:x1], k=1)

def overlay_single(ax, ct, mask, color, alpha=0.5):
    ax.imshow(ct, cmap="gray")
    overlay = np.zeros((*mask.shape, 4))
    overlay[..., :3] = color
    overlay[..., 3] = mask * alpha
    ax.imshow(overlay)
    ax.axis("off")

# ============================================================
# Plot & Save
# ============================================================

def plot_and_save_sample(ct, gt, pred, uid, slice_ids, case_idx, target_class, axis=2, save_dir="figures"):
    save_dir = Path(save_dir); save_dir.mkdir(parents=True, exist_ok=True)
    gt_bin, pred_bin = gt > 0, pred > 0
    bbox = compute_square_bbox(gt_bin | pred_bin, axis)

    fig, axes = plt.subplots(len(slice_ids), 2, figsize=(6, 3 * len(slice_ids)), dpi=150)
    if len(slice_ids) == 1: axes = axes[None, :]

    # Top Column Titles
    axes[0, 0].set_title("Ground Truth", fontsize=14, fontweight="bold", pad=10, color=COLOR_GT)
    axes[0, 1].set_title("Model Prediction", fontsize=14, fontweight="bold", pad=10, color=COLOR_PRED)

    for i, z in enumerate(slice_ids):
        sl_idx = [slice(None)] * 3
        sl_idx[axis] = z
        
        ct_sl = crop_and_rotate(window_ct_hu(ct[tuple(sl_idx)]), bbox)
        gt_sl = crop_and_rotate(gt_bin[tuple(sl_idx)], bbox)
        pr_sl = crop_and_rotate(pred_bin[tuple(sl_idx)], bbox)

        overlay_single(axes[i, 0], ct_sl, gt_sl, COLOR_GT)
        overlay_single(axes[i, 1], ct_sl, pr_sl, COLOR_PRED)

        axes[i, 0].text(-0.15, 0.5, f"Slice {z}", transform=axes[i, 0].transAxes, 
                        rotation=90, va="center", ha="right", fontsize=14, fontweight="bold")

    # Bottom Title: Case Number and Pathology
    path_name = PATH_NAMES.get(target_class, "Unknown")
    path_color = PATH_COLORS.get(target_class, "black")
    
    # Place text at the very bottom
    fig.text(0.5, 0.02, f"Case {case_idx}: {path_name}", 
             ha="center", va="bottom", fontsize=14, fontweight="bold", 
             color=path_color, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=3))

    plt.subplots_adjust(wspace=0.05, hspace=0.05, bottom=0.08)
    
    out_path = save_dir / f"qualitative_case{case_idx}_{uid}.pdf"
    plt.savefig(out_path, bbox_inches="tight", transparent=True)
    plt.show()

# ============================================================
# Interface
# ============================================================

def visualize_uids(uids, img_dir, gt_dir, pred_dir, label_csv, axis=2, slice_map=None, save_dir="figures"):
    labels_df = pd.read_csv(label_csv)
    labels_df['UID'] = labels_df['UID'].astype(str)

    for idx, uid in enumerate(uids, start=1): # case_idx starts at 1
        print(f"Processing Case {idx} (UID {uid})...")
        ct, gt, pred = load_case(uid, img_dir, gt_dir, pred_dir)
        
        # Get pathology class for coloring
        row = labels_df[labels_df['UID'] == str(uid)]
        target_class = int(row['target'].values[0]) if not row.empty else 0
        
        slices = slice_map[uid] if slice_map and uid in slice_map else [gt.shape[axis]//2]
        plot_and_save_sample(ct, gt, pred, uid, slices, idx, target_class, axis=axis, save_dir=save_dir)

# ============================================================
# Run
# ============================================================

visualize_uids(
    uids = [28, 344, 13, 239],
    slice_map = {28: [132, 154, 172], 344: [31, 35, 41], 13: [23, 42, 46], 239: [23, 25, 29]},
    img_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/imagesTs",
    gt_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labelsTs",
    pred_dir = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/predictionsTs",
    label_csv = "/data/colon_cancer/CC_Detection/raw_data/Dataset109_CC/labels.csv", 
    save_dir = "figures"
)